<a href="https://colab.research.google.com/github/ririfka08/netflix-analysis-dashboard/blob/main/netflix_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import Libraries
import pandas as pd
import plotly.express as px
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
# Load dataset
titles = pd.read_csv('/content/titles.csv')
credits = pd.read_csv('/content/credits.csv')

print("Shape:", titles.shape)
titles.head()

Shape: (5850, 15)


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600


In [ ]:
# Cek Struktur titles.csv
print(titles.info())
print("\n--- Missing values ---")
print(titles.isnull().sum())
print("\n--- Duplicate rows ---")
print(titles.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5850 entries, 0 to 5849
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    5850 non-null   object 
 1   title                 5849 non-null   object 
 2   type                  5850 non-null   object 
 3   description           5832 non-null   object 
 4   release_year          5850 non-null   int64  
 5   age_certification     3231 non-null   object 
 6   runtime               5850 non-null   int64  
 7   genres                5850 non-null   object 
 8   production_countries  5850 non-null   object 
 9   seasons               2106 non-null   float64
 10  imdb_id               5447 non-null   object 
 11  imdb_score            5368 non-null   float64
 12  imdb_votes            5352 non-null   float64
 13  tmdb_popularity       5759 non-null   float64
 14  tmdb_score            5539 non-null   float64
dtypes: float64(5), int64(

In [ ]:
# Cek Struktur credits.csv
print(credits.info())
print("\n--- Missing values ---")
print(credits.isnull().sum())
print("\n--- Duplicate rows ---")
print(credits.duplicated().sum())
credits.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77801 entries, 0 to 77800
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   person_id  77801 non-null  int64 
 1   id         77801 non-null  object
 2   name       77801 non-null  object
 3   character  68029 non-null  object
 4   role       77801 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.0+ MB
None

--- Missing values ---
person_id       0
id              0
name            0
character    9772
role            0
dtype: int64

--- Duplicate rows ---
0


,person_id,id,name,character,role
0,3748,tm84618,Robert De Niro,Travis Bickle,ACTOR
1,14658,tm84618,Jodie Foster,Iris Steensma,ACTOR
2,7064,tm84618,Albert Brooks,Tom,ACTOR
3,3739,tm84618,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,48933,tm84618,Cybill Shepherd,Betsy,ACTOR


In [ ]:
# Data Cleaning titles.csv
import ast

# 1. Drop duplicate
titles = titles.drop_duplicates()

# 2. Drop baris yang title atau imdb_score-nya kosong
titles = titles.dropna(subset=['title', 'imdb_score'])

# 3. Parse kolom genres & production_countries -> ubah dari string jadi list Python asli
#    Formatnya string "['drama', 'thriller']"
def parse_list_column(val):
    if pd.isna(val):
        return []
    try:
        result = ast.literal_eval(val)
        return result if isinstance(result, list) else []
    except (ValueError, SyntaxError):
        return []

titles['genres_list'] = titles['genres'].apply(parse_list_column)
titles['country_list'] = titles['production_countries'].apply(parse_list_column)

# 4. Buang judul yang genre atau negaranya kosong setelah parsing
titles = titles[titles['genres_list'].apply(len) > 0]

# 5. Buang runtime yang 0 (kemungkinan data error / durasi belum tercatat)
titles = titles[titles['runtime'] > 0]

print("Shape setelah cleaning:", titles.shape)
print("\nContoh genres_list:", titles['genres_list'].iloc[0])
print("Contoh country_list:", titles['country_list'].iloc[0])
print("\nDistribusi type:")
print(titles['type'].value_counts())


Shape setelah cleaning: (5351, 17)

Contoh genres_list: ['drama', 'crime']
Contoh country_list: ['US']

Distribusi type:
type
MOVIE    3427
SHOW     1924
Name: count, dtype: int64


In [ ]:
# Data Cleaning credits.csv
credits = credits.drop_duplicates()

print("Distribusi role:")
print(credits['role'].value_counts())
print("\nShape setelah cleaning:", credits.shape)


Distribusi role:
role
ACTOR       73251
DIRECTOR     4550
Name: count, dtype: int64

Shape setelah cleaning: (77801, 5)


In [ ]:
# JOIN titles + credits
# Join key-nya adalah kolom 'id' yang ada di kedua tabel

merged = titles.merge(credits, on='id', how='left')
print("Shape titles:", titles.shape)
print("Shape credits:", credits.shape)
print("Shape setelah merge:", merged.shape)
merged.head()

Shape titles: (5351, 17)
Shape credits: (77801, 5)
Shape setelah merge: (74114, 21)


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,genres_list,country_list,person_id,name,character,role
0,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,"[drama, crime]",[US],3748.0,Robert De Niro,Travis Bickle,ACTOR
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,"[drama, crime]",[US],14658.0,Jodie Foster,Iris Steensma,ACTOR
2,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,"[drama, crime]",[US],7064.0,Albert Brooks,Tom,ACTOR
3,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,"[drama, crime]",[US],3739.0,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,"[drama, crime]",[US],48933.0,Cybill Shepherd,Betsy,ACTOR


In [ ]:
# Explode Genre (1 baris = 1 genre per title)
# Ini penting biar bisa analisis per genre, sama kayak project buku

titles_exploded = titles.explode('genres_list')
titles_exploded['genres_list'] = titles_exploded['genres_list'].str.strip()
titles_exploded.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,genres_list,country_list
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,drama,[US]
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,crime,[US]
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300,drama,[US]
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300,action,[US]
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300,thriller,[US]


In [ ]:
# Analysis #1 - Genre Underserved (rating tinggi, jumlah sedikit)
# Jawab pertanyaan: genre apa yang berkualitas tapi kontennya masih sedikit -> peluang ekspansi

genre_stats = titles_exploded.groupby('genres_list').agg(
    avg_rating=('imdb_score', 'mean'),
    total_titles=('title', 'count')
).reset_index()
genre_stats = genre_stats[genre_stats['total_titles'] >= 20]
genre_stats = genre_stats.sort_values(['avg_rating', 'total_titles'], ascending=[False, True])

print(genre_stats.head(15))

fig1 = px.scatter(
    genre_stats, x='total_titles', y='avg_rating',
    text='genres_list', size='total_titles',
    title='Genre Content Volume vs IMDb Rating',
    labels={'total_titles': 'Jumlah Konten', 'avg_rating': 'Rating Rata-rata'}
)
fig1.update_traces(textposition='top center')
fig1.show()


      genres_list  avg_rating  total_titles
9         history    7.128740           254
17            war    7.065806           155
4   documentation    7.009557           858
1       animation    6.701749           629
15          sport    6.652353           170
3           crime    6.649173           907
5           drama    6.622971          2821
8         fantasy    6.558158           619
14          scifi    6.545179           560
6        european    6.529673           428
11          music    6.512295           244
18        western    6.484615            39
0          action    6.417977          1107
12        reality    6.415207           217
2          comedy    6.393631          2214


In [ ]:
# Analysis #2 - Movie vs TV Show
# Jawab pertanyaan: mana yang secara rata-rata lebih diterima penonton?

type_comparison = titles.groupby('type').agg(
    avg_rating=('imdb_score', 'mean'),
    avg_popularity=('tmdb_popularity', 'mean'),
    total=('title', 'count')
).reset_index()
print(type_comparison)

fig2 = px.bar(type_comparison, x='type', y='avg_rating',
              color='type', title='Movie vs TV Show - Average IMDb Score')
fig2.show()


    type  avg_rating  avg_popularity  total
0  MOVIE    6.246250       20.923813   3427
1   SHOW    6.977599       28.132218   1924


In [ ]:
# Analysis #3 - Negara Produksi Paling Efisien
# Jawab pertanyaan: negara mana yang rating tinggi meski jumlah produksinya sedikit?

country_exploded = titles.explode('country_list')
country_exploded['country_list'] = country_exploded['country_list'].str.strip()

country_stats = country_exploded.groupby('country_list').agg(
    avg_rating=('imdb_score', 'mean'),
    total_titles=('title', 'count')
).reset_index()
country_stats = country_stats[country_stats['total_titles'] >= 10]
country_stats = country_stats.sort_values('avg_rating', ascending=False)

print(country_stats.head(15))


   country_list  avg_rating  total_titles
57           KR    7.225381           197
44           IL    7.061111            18
52           JP    6.974131           259
43           IE    6.880000            15
81           PS    6.866667            15
26           DK    6.853571            28
33           GB    6.804497           378
84           QA    6.780000            10
20           CN    6.769149            94
87           RU    6.746667            15
73           NO    6.681818            22
97           TW    6.679310            58
59           LB    6.645455            33
6            AU    6.613415            82
17           CH    6.569231            13


In [ ]:
# Analysis #4 - Durasi vs Rating
# Jawab pertanyaan: apakah ada "sweet spot" durasi ideal?

fig3 = px.scatter(titles, x='runtime', y='imdb_score', color='type',
                   title='Runtime vs IMDb Score',
                   labels={'runtime': 'Durasi (menit)', 'imdb_score': 'IMDb Score'})
fig3.show()



In [ ]:
# Analysis #5 - Tren Rating per Tahun Rilis
# Jawab pertanyaan: apakah kualitas konten Netflix membaik/menurun dari waktu ke waktu?

yearly = titles.groupby('release_year')['imdb_score'].mean().reset_index()
# Filter tahun yang datanya cukup banyak biar nggak bias sama tahun dengan 1-2 judul doang
year_counts = titles['release_year'].value_counts()
valid_years = year_counts[year_counts >= 10].index
yearly = yearly[yearly['release_year'].isin(valid_years)]

fig4 = px.line(yearly, x='release_year', y='imdb_score',
                title='Average IMDb Score by Release Year',
                labels={'release_year': 'Tahun Rilis', 'imdb_score': 'Rating Rata-rata'})
fig4.show()

In [ ]:
# Analysis #6 - Cast Paling Sering di Konten Rating Tinggi (pakai data merged)
# Ini analisis tambahan yang manfaatin join tabel - siapa aktor yang paling sering
# muncul di judul-judul dengan rating tinggi (di atas 7.5)

high_rated = merged[merged['imdb_score'] >= 7.5]
top_actors = high_rated[high_rated['role'] == 'ACTOR'].groupby('name').agg(
    appearances=('id', 'count'),
    avg_rating_of_titles=('imdb_score', 'mean')
).reset_index()
top_actors = top_actors[top_actors['appearances'] >= 3]
top_actors = top_actors.sort_values('appearances', ascending=False)

print(top_actors.head(15))


                    name  appearances  avg_rating_of_titles
10246        Terry Jones           12              8.025000
10            Aamir Khan           11              8.109091
2977           Eric Idle           11              8.009091
7237       Michael Palin           11              8.009091
10245      Terry Gilliam           10              8.000000
4757         John Cleese           10              7.930000
11123          Yuki Kaji            9              7.966667
3541      Graham Chapman            9              8.100000
10134   Takahiro Sakurai            8              7.850000
1466      Brijendra Kala            8              7.825000
9266   Samuel L. Jackson            7              7.914286
162         Ahn Nae-sang            7              8.000000
8487         R. Madhavan            7              8.142857
2164         Daisuke Ono            7              8.042857
1363         Boman Irani            7              7.800000


In [ ]:
# Save Cleaned Data (buat dashboard nanti)
titles.to_csv('titles_cleaned.csv', index=False)
titles_exploded.to_csv('titles_exploded.csv', index=False)
country_exploded.to_csv('titles_country_exploded.csv', index=False)

from google.colab import files
files.download('titles_cleaned.csv')
files.download('titles_exploded.csv')
files.download('titles_country_exploded.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>